In [5]:

import sys
import os
from tqdm import tqdm
import shutil

# Add the src directory to the Python path
sys.path.append(os.path.abspath("../src"))
from data.prepare_dataset import extract_patient_id

# Add the src directory to the Python path
def save_dataset_images(dataset, save_dir, class_names, max_images=None):
    """Save images from a dataset to a directory with class/patient/image structure.
    
    Args:
        dataset: A NeurofluxDataset instance
        save_dir (str): Directory where to save the images
        class_names (list): List of class names corresponding to labels
        max_images (int, optional): Maximum number of images to save. If None, saves all images.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Determine how many images to save
    n_images = len(dataset) if max_images is None else min(max_images, len(dataset))
    
    # Keep track of saved images per class
    saved_per_class = {class_name: 0 for class_name in class_names}
    
    for idx in tqdm(range(n_images), desc="Saving images"):
        # Get the image path and label
        img_path = dataset.image_paths[idx]
        label = dataset.labels[idx]
        
        # Get class name and patient ID
        class_name = class_names[label]
        patient_id = extract_patient_id(os.path.basename(img_path))
        
        # Skip if patient ID couldn't be extracted
        if not patient_id:
            print(f"Warning: Could not extract patient ID from {os.path.basename(img_path)}, using 'unknown'")
            patient_id = "unknown"
        
        # Create class and patient directories
        class_dir = os.path.join(save_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)
        
        patient_dir = os.path.join(class_dir, patient_id)
        os.makedirs(patient_dir, exist_ok=True)
        
        # Copy the image to the destination
        img_name = os.path.basename(img_path)
        dst_path = os.path.join(patient_dir, img_name)
        
        try:
            # Option 1: Copy the file directly (faster but requires same file system)
            shutil.copy2(img_path, dst_path)
            saved_per_class[class_name] += 1
            
            # Option 2: Open and save (slower but more reliable across file systems)
            # img = Image.open(img_path)
            # img.save(dst_path)
        except Exception as e:
            print(f"Error processing image {img_path}: {e}")
    
    # Print summary of saved images
    print("\nSaved images per class:")
    for class_name, count in saved_per_class.items():
        print(f"  {class_name}: {count} images")

In [ ]:
import os
import yaml

# Add the src directory to the Python path
sys.path.append(os.path.abspath("../src"))
from data.prepare_dataset import load_data, extract_patient_id

# Configuration file path - you can modify this
config_file = r'../configs/config_debug.yaml'

# Specify the save locations - you can modify these paths as needed
save_dirs = {
    'train': "debug_images/train",
    'val': "debug_images/val",
    'test': "debug_images/test"
}

# Load configuration
with open(config_file, "r") as f:
    config = yaml.safe_load(f)
    
# Load datasets
print("Loading datasets...")
train_dataset, val_dataset, test_dataset = load_data(config)

# Create directories if they don't exist
for dir_path in save_dirs.values():
    os.makedirs(dir_path, exist_ok=True)

class_names = config["general"]["class_names"]
# Save images from each dataset
#print("\nSaving training dataset images...")
#save_dataset_images(train_dataset, save_dirs['train'], class_names, max_images=20)

#print("\nSaving validation dataset images...")
#save_dataset_images(val_dataset, save_dirs['val'], class_names, max_images=10)

print("\nSaving test dataset images...")
save_dataset_images(test_dataset, save_dirs['test'], class_names, max_images=None)

print("\nAll images have been saved successfully!")


Loading datasets...
CSV saved at: debug_csv/train_metadata.csv
CSV saved at: debug_csv/val_metadata.csv
CSV saved at: debug_csv/test_metadata.csv

Saving test dataset images...


TypeError: save_dataset_images() missing 1 required positional argument: 'class_names'